# Treinamento EfficientNetB0 — Técnica 1: blocos finais

Este notebook executa a **EfficientNetB0** na **Técnica 1 — Fine-tuning dos blocos finais**.

Configuração principal:

- `IMG_SIZE = (448, 448)`;
- `BATCH_SIZE = 24`;
- `NUM_WORKERS = 8`;
- `CLASS_WEIGHT_MODE = "sqrt"`;
- `CrossEntropyLoss` com `LABEL_SMOOTHING = 0.015`;
- `FINE_TUNING_SCOPE = "features_last2"`;
- fine-tuning em `features.7 + features.8 + classifier`.

Os resultados serão salvos na nova estrutura:

- `results/metrics/t1_blocos_finais/efficientnetb0/`;
- `results/figures/t1_blocos_finais/efficientnetb0/`;
- `models/t1_blocos_finais/efficientnetb0/`.


In [ ]:
# CÉLULA 02 - Importações e verificação de GPU

import os
# Reduz fragmentação de memória CUDA em testes com imagens maiores.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# CÉLULA 03 - Configurações gerais do experimento

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Identificação da técnica experimental.
TECHNIQUE_ID = "t1"
TECHNIQUE_NAME = "blocos_finais"
TECHNIQUE_DESCRIPTION = "Fine-tuning restrito aos blocos finais da arquitetura."

# Configuração padronizada para os experimentos principais.
# Se ocorrer CUDA out of memory, reduza apenas BATCH_SIZE para 16 ou 12.
IMG_SIZE = (448, 448)
BATCH_SIZE = 24
NUM_CLASSES = 5

NUM_WORKERS = 8
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0

EPOCHS_HEAD = 10
EPOCHS_FINE_TUNING = 40

HEAD_LEARNING_RATE = 1e-3
FINE_TUNING_LEARNING_RATE = 1e-5

DROPOUT_RATE = 0.3
PATIENCE = 6

# Pesos de classe suavizados.
CLASS_WEIGHT_MODE = "sqrt"  # opções: "balanced", "sqrt", "none"

# Função de perda usada na comparação principal.
LOSS_FUNCTION = "cross_entropy"  # opções: "cross_entropy", "focal"
FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.015

# Mantido desativado porque o sampler prejudicou o equilíbrio geral nos testes anteriores.
USE_WEIGHTED_SAMPLER = False

# Técnica 1 — blocos finais da EfficientNetB0.
# features_last2 ajusta features.7 + features.8 + classifier.
FINE_TUNING_SCOPE = "features_last2"  # opções: "features_last2", "features_last3"

MODEL_NAME = "EfficientNetB0"
MODEL_KEY = "efficientnetb0"
FRAMEWORK_KEY = "pytorch_gpu"

EXPERIMENT_NAME = "t1_blocos_finais_448_bs24_workers8_sqrt_ce_ls0015_ft_features_last2_lr1e5"
MODEL_OUTPUT_KEY = f"{MODEL_KEY}_{FRAMEWORK_KEY}_{EXPERIMENT_NAME}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo usado:", device)
print("Configuração:", {
    "TECHNIQUE_ID": TECHNIQUE_ID,
    "TECHNIQUE_NAME": TECHNIQUE_NAME,
    "IMG_SIZE": IMG_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "NUM_WORKERS": NUM_WORKERS,
    "CLASS_WEIGHT_MODE": CLASS_WEIGHT_MODE,
    "LOSS_FUNCTION": LOSS_FUNCTION,
    "LABEL_SMOOTHING": LABEL_SMOOTHING,
    "USE_WEIGHTED_SAMPLER": USE_WEIGHTED_SAMPLER,
    "FINE_TUNING_SCOPE": FINE_TUNING_SCOPE,
})


In [ ]:
# CÉLULA 04 - Caminhos do projeto

cwd = Path.cwd().resolve()

# Funciona tanto se o notebook for executado da raiz do projeto quanto de dentro de notebooks/
if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

SPLITS_DIR = PROJECT_DIR / "data" / "splits"
RAW_IMAGES_DIR = PROJECT_DIR / "data" / "raw" / "train_images"

RESULTS_DIR = PROJECT_DIR / "results"
TECHNIQUE_DIR_NAME = f"{TECHNIQUE_ID}_{TECHNIQUE_NAME}"

METRICS_DIR = RESULTS_DIR / "metrics" / TECHNIQUE_DIR_NAME / MODEL_KEY
FIGURES_DIR = RESULTS_DIR / "figures" / TECHNIQUE_DIR_NAME / MODEL_KEY
MODELS_DIR = PROJECT_DIR / "models" / TECHNIQUE_DIR_NAME / MODEL_KEY

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_OUTPUT_DIR = MODELS_DIR / MODEL_OUTPUT_KEY
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("SPLITS_DIR:", SPLITS_DIR)
print("RAW_IMAGES_DIR:", RAW_IMAGES_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("MODEL_OUTPUT_DIR:", MODEL_OUTPUT_DIR)


In [ ]:
# CÉLULA 05 - Carregar splits e corrigir caminhos

train_df = pd.read_csv(SPLITS_DIR / "train_split.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_split.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_split.csv")

def corrigir_caminhos(df):
    df = df.copy()
    df["image_path"] = df["id_code"].apply(
        lambda image_id: str((RAW_IMAGES_DIR / f"{image_id}.png").resolve())
    )
    return df

train_df = corrigir_caminhos(train_df)
val_df = corrigir_caminhos(val_df)
test_df = corrigir_caminhos(test_df)

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)

train_df.head()

In [ ]:
# CÉLULA 06 - Verificar imagens ausentes

def verificar_imagens(df, nome):
    missing = df[~df["image_path"].apply(lambda path: Path(path).exists())]
    print(f"{nome}:")
    print("Total:", len(df))
    print("Imagens ausentes:", len(missing))
    if len(missing) > 0:
        display(missing.head())
        raise FileNotFoundError(f"Existem imagens ausentes em {nome}.")

verificar_imagens(train_df, "Treino")
verificar_imagens(val_df, "Validação")
verificar_imagens(test_df, "Teste")

In [ ]:
# CÉLULA 07 - Nomes das classes

class_names = {
    0: "Sem retinopatia",
    1: "Retinopatia leve",
    2: "Retinopatia moderada",
    3: "Retinopatia severa",
    4: "Retinopatia proliferativa",
}

class_labels = [class_names[i] for i in range(NUM_CLASSES)]

print(class_labels)

In [ ]:
# CÉLULA 08 - Transformações de treino, validação e teste

# Normalização padrão ImageNet, compatível com modelos pré-treinados do torchvision.
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

In [ ]:
# CÉLULA 09 - Dataset e DataLoader

class AptosDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        label = int(row["diagnosis"])

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = AptosDataset(train_df, transform=train_transform)
val_dataset = AptosDataset(val_df, transform=eval_transform)
test_dataset = AptosDataset(test_df, transform=eval_transform)

sampler = None

if USE_WEIGHTED_SAMPLER:
    train_labels = train_df["diagnosis"].astype(int).values
    class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    class_sample_weights = 1.0 / class_counts

    sample_weights = np.array([
        class_sample_weights[label]
        for label in train_labels
    ])

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    print("WeightedRandomSampler ativado.")
    print("Contagem por classe:", class_counts)
else:
    print("WeightedRandomSampler desativado. Usando shuffle=True no treino.")

# Se sampler estiver ativo, não pode usar shuffle=True.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=(sampler is None),
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

images, labels = next(iter(train_loader))
print("Batch imagens:", images.shape)
print("Batch rótulos:", labels.shape)
print("Dtype imagens:", images.dtype)
print("Dtype rótulos:", labels.dtype)


In [ ]:
# CÉLULA 10 - Pesos por classe e função de perda

classes = np.unique(train_df["diagnosis"].values)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"].values
)

class_weights_balanced = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, class_weights_values)
}

class_weights_sqrt = {
    class_id: float(np.sqrt(weight))
    for class_id, weight in class_weights_balanced.items()
}

if CLASS_WEIGHT_MODE == "balanced":
    class_weights = class_weights_balanced
elif CLASS_WEIGHT_MODE == "sqrt":
    class_weights = class_weights_sqrt
elif CLASS_WEIGHT_MODE == "none":
    class_weights = None
else:
    raise ValueError("CLASS_WEIGHT_MODE deve ser 'balanced', 'sqrt' ou 'none'.")

print("Class weights balanceados:")
print(class_weights_balanced)

print("\nClass weights suavizados:")
print(class_weights_sqrt)

print("\nClass weights usados neste experimento:")
print(class_weights)

if class_weights is not None:
    weight_tensor = torch.tensor(
        [class_weights[i] for i in range(NUM_CLASSES)],
        dtype=torch.float32,
        device=device
    )
else:
    weight_tensor = None


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs,
            targets,
            weight=self.alpha,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss

        return loss.mean()


if LOSS_FUNCTION == "cross_entropy":
    criterion = nn.CrossEntropyLoss(
        weight=weight_tensor,
        label_smoothing=LABEL_SMOOTHING
    )
elif LOSS_FUNCTION == "focal":
    criterion = FocalLoss(alpha=weight_tensor, gamma=FOCAL_GAMMA)
else:
    raise ValueError("LOSS_FUNCTION deve ser 'cross_entropy' ou 'focal'.")

print("\nFunção de perda usada:", LOSS_FUNCTION)
if LOSS_FUNCTION == "cross_entropy":
    print("Label smoothing:", LABEL_SMOOTHING)
if LOSS_FUNCTION == "focal":
    print("Focal gamma:", FOCAL_GAMMA)


In [ ]:
# CÉLULA 11 - Criar modelo EfficientNetB0 e treinar somente a cabeça inicialmente

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Congelar todas as camadas inicialmente.
for param in model.parameters():
    param.requires_grad = False

# Trocar a cabeça final.
# No torchvision, a EfficientNetB0 possui model.classifier = Sequential(Dropout, Linear).
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT_RATE),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(device)

# Apenas a cabeça treina na primeira etapa.
optimizer_head = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=HEAD_LEARNING_RATE
)

scheduler_head = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_head,
    mode="min",
    factor=0.2,
    patience=3,
    min_lr=1e-7
)

print(model.classifier)

# Teste de forward
model.eval()
with torch.no_grad():
    sample_images = images.to(device)
    preds = model(sample_images)
    print("Predições:", preds.shape)


In [ ]:
# CÉLULA 12 - Funções de treino, validação, early stopping e avaliação

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(predicted.cpu().numpy().tolist())

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds)


def fit_phase(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs,
    checkpoint_path,
    phase_name,
    patience=PATIENCE,
):
    history = []

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc, _, _ = evaluate(
            model, val_loader, criterion, device
        )

        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "phase": phase_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "learning_rate": current_lr,
        }
        history.append(row)

        print(
            f"[{phase_name}] Época {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={current_lr:.2e}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  Melhor modelo salvo em: {checkpoint_path}")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping na fase {phase_name}. Melhor época: {best_epoch}")
            break

    phase_time = time.time() - start_time

    # Restaurar melhor checkpoint da fase.
    try:
        state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)

    return history, phase_time

In [ ]:
# CÉLULA 13 - Treinar a cabeça do modelo

checkpoint_head_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_head_best.pt"

history_head, head_training_time = fit_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_head,
    scheduler=scheduler_head,
    device=device,
    epochs=EPOCHS_HEAD,
    checkpoint_path=checkpoint_head_path,
    phase_name="head",
)

print(f"Tempo de treinamento da cabeça: {head_training_time:.2f} segundos")

In [ ]:
# CÉLULA 14 - Configurar fine-tuning

# Opções:
# - FINE_TUNING_SCOPE = "features_last2": ajusta features.7 + features.8 + classifier.
# - FINE_TUNING_SCOPE = "features_last3": ajusta features.6 + features.7 + features.8 + classifier.
#   Esta é a opção equivalente ao denseblock3_4 usado na DenseNet121 final.

for name, param in model.named_parameters():
    if FINE_TUNING_SCOPE == "features_last2":
        param.requires_grad = (
            name.startswith("features.7")
            or name.startswith("features.8")
            or name.startswith("classifier")
        )
    elif FINE_TUNING_SCOPE == "features_last3":
        param.requires_grad = (
            name.startswith("features.6")
            or name.startswith("features.7")
            or name.startswith("features.8")
            or name.startswith("classifier")
        )
    else:
        raise ValueError("FINE_TUNING_SCOPE deve ser 'features_last2' ou 'features_last3'.")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Escopo do fine-tuning:", FINE_TUNING_SCOPE)
print("Parâmetros treináveis:", trainable_params)
print("Parâmetros totais:", total_params)

optimizer_fine = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FINE_TUNING_LEARNING_RATE
)

scheduler_fine = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fine,
    mode="min",
    factor=0.2,
    patience=3,
    min_lr=1e-7
)


In [ ]:
# CÉLULA 15 - Treinar a etapa de fine-tuning

checkpoint_fine_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_fine_best.pt"

history_fine, fine_tuning_time = fit_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_fine,
    scheduler=scheduler_fine,
    device=device,
    epochs=EPOCHS_FINE_TUNING,
    checkpoint_path=checkpoint_fine_path,
    phase_name="fine_tuning",
)

print(f"Tempo de fine-tuning: {fine_tuning_time:.2f} segundos")

In [ ]:
# CÉLULA 16 - Consolidar e salvar histórico

history_df = pd.DataFrame(history_head + history_fine)
history_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_history.csv"
history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

print("Histórico salvo em:", history_path)
history_df.tail()

In [ ]:
# CÉLULA 17 - Gerar gráficos de loss e acurácia

plt.figure(figsize=(8, 5))
plt.plot(history_df.index + 1, history_df["train_loss"], label="Loss treino")
plt.plot(history_df.index + 1, history_df["val_loss"], label="Loss validação")
plt.title(f"Loss durante o treinamento - {MODEL_NAME}")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_loss.png"
plt.savefig(loss_fig_path, dpi=300)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(history_df.index + 1, history_df["train_accuracy"], label="Acurácia treino")
plt.plot(history_df.index + 1, history_df["val_accuracy"], label="Acurácia validação")
plt.title(f"Acurácia durante o treinamento - {MODEL_NAME}")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(True)
plt.tight_layout()

acc_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_accuracy.png"
plt.savefig(acc_fig_path, dpi=300)
plt.show()

print("Figuras salvas em:")
print(loss_fig_path)
print(acc_fig_path)

In [ ]:
# CÉLULA 18 - Avaliação final no conjunto de teste

test_loss, test_accuracy, y_true, y_pred = evaluate(
    model, test_loader, criterion, device
)

print("Loss no teste:", test_loss)
print("Acurácia no teste:", test_accuracy)

In [ ]:
# CÉLULA 19 - Gerar relatório de classificação

report = classification_report(
    y_true,
    y_pred,
    target_names=class_labels,
    digits=4
)

print(report)

report_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_classification_report.txt"
report_path.write_text(report, encoding="utf-8")

print("Relatório salvo em:", report_path)

In [ ]:
# CÉLULA 20 - Gerar matriz de confusão

cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=class_labels,
    columns=class_labels
)

cm_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_confusion_matrix.csv"
cm_df.to_csv(cm_path, encoding="utf-8-sig")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_labels
)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap="viridis", values_format="d")
plt.title(f"Matriz de Confusão - {MODEL_NAME}")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

cm_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_confusion_matrix.png"
plt.savefig(cm_fig_path, dpi=300)
plt.show()

print("Matriz salva em:", cm_path)
print("Figura salva em:", cm_fig_path)

In [ ]:
# CÉLULA 21 - Salvar modelo final e resumo de métricas

final_model_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_final.pt"
torch.save(model.state_dict(), final_model_path)

summary_metrics = {
    "model": MODEL_NAME,
    "framework": "PyTorch",
    "technique_id": TECHNIQUE_ID,
    "technique_name": TECHNIQUE_NAME,
    "technique_description": TECHNIQUE_DESCRIPTION,
    "experiment_name": EXPERIMENT_NAME,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "epochs_head_configured": EPOCHS_HEAD,
    "epochs_fine_tuning_configured": EPOCHS_FINE_TUNING,
    "epochs_completed_total": int(len(history_df)),
    "batch_size": BATCH_SIZE,
    "image_size": list(IMG_SIZE),
    "dropout_rate": DROPOUT_RATE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "persistent_workers": PERSISTENT_WORKERS,
    "class_weight_mode": CLASS_WEIGHT_MODE,
    "class_weights": class_weights,
    "loss_function": LOSS_FUNCTION,
    "focal_gamma": FOCAL_GAMMA if LOSS_FUNCTION == "focal" else None,
    "label_smoothing": LABEL_SMOOTHING if LOSS_FUNCTION == "cross_entropy" else None,
    "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
    "split_strategy": "60% treino / 20% validação / 20% teste",
    "preprocess_input": "torchvision.transforms.Normalize(mean=ImageNet, std=ImageNet)",
    "data_augmentation": {
        "Resize": list(IMG_SIZE),
        "RandomRotation": 8,
        "RandomAffine_translate": [0.03, 0.03],
        "RandomAffine_scale": [0.95, 1.05],
        "RandomHorizontalFlip": 0.5,
    },
    "run_fine_tuning": True,
    "fine_tuning_scope": FINE_TUNING_SCOPE,
    "trainable_params": int(trainable_params),
    "total_params": int(total_params),
    "trainable_percent": float(100 * trainable_params / total_params),
    "fine_tuning_layers": (
        "features.7 + features.8 + classifier"
        if FINE_TUNING_SCOPE == "features_last2"
        else "features.6 + features.7 + features.8 + classifier"
    ),
    "head_learning_rate": HEAD_LEARNING_RATE,
    "fine_tuning_learning_rate": FINE_TUNING_LEARNING_RATE,
    "head_training_time_seconds": float(head_training_time),
    "fine_tuning_time_seconds": float(fine_tuning_time),
    "gpu_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "final_model_path": str(final_model_path),
    "best_head_checkpoint_path": str(checkpoint_head_path),
    "best_fine_checkpoint_path": str(checkpoint_fine_path),
}

summary_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_summary_metrics.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=4, ensure_ascii=False)

print("Modelo final salvo em:", final_model_path)
print("Resumo salvo em:", summary_path)
summary_metrics


## Observações

Este notebook corresponde à **EfficientNetB0 — Técnica 1: blocos finais**.

Use principalmente:

- **CÉLULA 03**: hiperparâmetros e identificação da técnica;
- **CÉLULA 04**: nova estrutura de salvamento em `results/metrics`, `results/figures` e `models`;
- **CÉLULA 14**: escopo do fine-tuning (`features_last2` para Técnica 1);
- **CÉLULA 21**: salvamento do modelo e do resumo de métricas.

Para comparação metodológica:

- Técnica 1: `features.7 + features.8 + classifier`;
- Técnica 2: `features.6 + features.7 + features.8 + classifier`;
- Técnica 3: `features.4 + features.5 + features.6 + features.7 + features.8 + classifier`.
